# Przegląd jakości zdjęć EyePACS

Notebook do kalibracji progów czyszczenia danych. Korzysta z cache'u i metryk z `scripts/build_cache.py`:

```
python scripts/build_cache.py --src <raw/train> --labels <raw/trainLabels.csv> --dst <cache_768>
```

Plan:
1. Rozkłady metryk i ich zależność od aparatu (rozdzielczości).
2. Siatki zdjęć z obu krańców każdej metryki → oceniamy wzrokiem, gdzie zaczyna się „nie nadaje się".
3. Związek jakości z etykietami.
4. Szkic reguł odrzucania (`THRESHOLDS`) + losowe próbki odrzuconych zdjęć do weryfikacji.

Ścieżkę do cache'u można ustawić zmienną środowiskową `EYEPACS_CACHE`.

In [ ]:
import os
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)

CACHE_DIR = Path(os.environ.get('EYEPACS_CACHE', '~/data/eyepacs/cache_768')).expanduser()
IMAGES_DIR = CACHE_DIR / 'images'

metrics = pd.read_csv(CACHE_DIR / 'metrics.csv')
metrics['binary_label'] = (metrics['level'] > 0).astype(int)
metrics['patient_id'] = metrics['image'].str.replace(r'_(left|right)$', '', regex=True)

print('cache:', CACHE_DIR)
print('zdjęć:', len(metrics))
print('nieczytelnych:', (~metrics['readable']).sum())
print('bez wykrytego dna oka (NaN):', metrics['mean_brightness'].isna().sum())

## Opis metryk

| metryka | co mierzy | podejrzane gdy |
|---|---|---|
| `mask_frac` | udział koła dna oka w oryginalnym kadrze | bardzo mały → brak/ucięte dno oka |
| `bbox_h`, `bbox_aspect` | wysokość i proporcje koła | koło ucięte, przesunięte |
| `mean_brightness` | średnia jasność wewnątrz koła | za ciemne / prześwietlone |
| `contrast` | odchylenie std jasności w kole | „mleczne", płaskie zdjęcie |
| `frac_dark`, `frac_bright` | odsetek pikseli bardzo ciemnych / przepalonych w kole | duże ciemne plamy, odblaski |
| `sharpness` | wariancja Laplasjanu kanału G w kole (po resize do 768) | rozmazanie, nieostrość |
| `sharpness_z` | `sharpness` znormalizowana w obrębie aparatu (patrz niżej) | ≪ 0 |

In [ ]:
num_cols = ['mask_frac', 'bbox_h', 'bbox_aspect', 'mean_brightness', 'contrast',
            'frac_dark', 'frac_bright', 'sharpness']

fig, axes = plt.subplots(2, 4, figsize=(18, 7))
for ax, col in zip(axes.flat, num_cols):
    values = metrics[col].dropna()
    log = col in ('sharpness', 'frac_dark', 'frac_bright')
    if log:
        values = np.log10(values + 1e-4)
    ax.hist(values, bins=100)
    ax.set_title(f'log10({col})' if log else col)
plt.tight_layout()
plt.show()

metrics[num_cols].describe(percentiles=[.001, .01, .05, .5, .95, .99, .999]).T.round(3)

## Ostrość zależy od aparatu

EyePACS zbierano wieloma aparatami — rozdzielczość oryginału (`orig_w`) dobrze je rozróżnia.
Mediana `sharpness` różni się między aparatami nawet 3× (np. 2560 px ≈ 13, 2592 px ≈ 38), więc
jeden globalny próg wyciąłby całe aparaty zamiast nieostrych zdjęć.

Dlatego liczymy `sharpness_z`: odporny z-score `log(sharpness)` w obrębie grupy `orig_w`
(mediana i MAD zamiast średniej i std). Rzadkie rozdzielczości (< 100 zdjęć) wrzucamy do wspólnej grupy.

In [ ]:
by_cam = (
    metrics.groupby('orig_w')
    .agg(n=('image', 'size'),
         sharpness_med=('sharpness', 'median'),
         brightness_med=('mean_brightness', 'median'),
         dr_rate=('binary_label', 'mean'))
    .sort_values('n', ascending=False)
    .round(3)
)
display(by_cam)

camera_group = metrics['orig_w'].where(metrics['orig_w'].map(metrics['orig_w'].value_counts()) >= 100, -1)
metrics['camera_group'] = camera_group

log_sharp = np.log(metrics['sharpness'] + 1e-3)
grp = log_sharp.groupby(metrics['camera_group'])
med = grp.transform('median')
mad = grp.transform(lambda s: (s - s.median()).abs().median())
metrics['sharpness_z'] = (log_sharp - med) / (1.4826 * mad)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(metrics['sharpness_z'].dropna(), bins=150)
ax.set_title('sharpness_z (w obrębie aparatu)')
plt.show()

## Narzędzia do przeglądania

- `show_grid(df, title, value_col)` — siatka miniatur z podpisem: nazwa, klasa, wartość metryki.
- `show_extremes(col, n)` — n najniższych i n najwyższych wartości metryki.
- `show_around(col, value, n)` — n zdjęć najbliżej zadanej wartości. **Tym kalibrujemy próg**:
  oglądamy zdjęcia tuż przy kandydującym progu i sprawdzamy, czy jeszcze są OK.

In [ ]:
def load_thumb(image_name: str, size: int = 256) -> np.ndarray:
    image = cv2.imread(str(IMAGES_DIR / f'{image_name}.jpeg'))
    if image is None:
        return np.zeros((size, size, 3), dtype=np.uint8)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    return cv2.resize(image, (size, size), interpolation=cv2.INTER_AREA)


def show_grid(df: pd.DataFrame, title: str = '', value_col: str | None = None,
              cols: int = 8, thumb: int = 256) -> None:
    if len(df) == 0:
        print(f'{title}: brak zdjęć')
        return
    rows = int(np.ceil(len(df) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.2, rows * 2.45))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes:
        ax.axis('off')
    for ax, row in zip(axes, df.itertuples()):
        ax.imshow(load_thumb(row.image, thumb))
        label = f'{row.image}\nklasa {row.level}'
        if value_col is not None:
            label += f' | {getattr(row, value_col):.3g}'
        ax.set_title(label, fontsize=7)
    fig.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.show()


def show_extremes(col: str, n: int = 16) -> None:
    valid = metrics.dropna(subset=[col]).sort_values(col)
    show_grid(valid.head(n), f'{col}: NAJNIŻSZE', col)
    show_grid(valid.tail(n).iloc[::-1], f'{col}: NAJWYŻSZE', col)


def show_around(col: str, value: float, n: int = 16) -> None:
    valid = metrics.dropna(subset=[col])
    idx = (valid[col] - value).abs().nsmallest(n).index
    subset = valid.loc[idx].sort_values(col)
    below = (valid[col] < value).sum()
    show_grid(subset, f'{col} ≈ {value}  (poniżej tej wartości: {below} zdjęć, {below / len(valid):.1%})', col)

## Zdjęcia bez wykrytego dna oka / całkiem puste

In [ ]:
empty = metrics[metrics['mean_brightness'].isna() | (metrics['mask_frac'] < 0.2)]
show_grid(empty.sort_values('mask_frac'), f'mask_frac < 0.2 lub brak maski: {len(empty)}', 'mask_frac')

## Ostrość (`sharpness_z`)

In [ ]:
show_extremes('sharpness_z', n=24)

In [ ]:
# Kalibracja progu: zdjęcia wokół kandydujących wartości
for value in (-3.0, -2.5, -2.0):
    show_around('sharpness_z', value, n=16)

## Jasność (`mean_brightness`)

In [ ]:
show_extremes('mean_brightness', n=24)

In [ ]:
for value in (20, 25, 30):
    show_around('mean_brightness', value, n=16)

## Kontrast (`contrast`)

In [ ]:
show_extremes('contrast', n=24)

In [ ]:
for value in (6, 8, 10):
    show_around('contrast', value, n=16)

## Ciemne plamy i przepalenia (`frac_dark`, `frac_bright`)

In [ ]:
show_extremes('frac_dark', n=16)
show_extremes('frac_bright', n=16)

In [ ]:
show_around('frac_dark', 0.3, n=16)
show_around('frac_bright', 0.1, n=16)

## Geometria koła (`mask_frac`, `bbox_h`, `bbox_aspect`)

In [ ]:
show_extremes('mask_frac', n=16)
show_extremes('bbox_aspect', n=16)
valid = metrics.dropna(subset=['bbox_h']).sort_values('bbox_h')
show_grid(valid.head(16), 'bbox_h: NAJNIŻSZE (koło ucięte z góry/dołu?)', 'bbox_h')

## Duplikaty — uwaga

`phash` (8×8) daje ~250 grup „identycznych" hashy, ale **wszystkie łączą różnych pacjentów**
i po obejrzeniu to po prostu podobnie skadrowane zdjęcia (tarcza nerwu w tym samym miejscu).
Perceptual hash jest za grubym narzędziem dla zdjęć dna oka — **nie używamy go do odrzucania**.
Ewentualne prawdziwe duplikaty można później wykryć embeddingami CNN.

In [ ]:
dup_groups = metrics[metrics['phash'].duplicated(keep=False)].groupby('phash')
print('grup:', dup_groups.ngroups)
print('grup z jednym pacjentem:', sum(g['patient_id'].nunique() == 1 for _, g in dup_groups))
sample = pd.concat([g for _, g in list(dup_groups)[:8]])
show_grid(sample, 'przykładowe grupy z identycznym phash (kolejne pary)', cols=8)

## Jakość a etykiety

Ważne pytanie: czy słaba jakość jest skorelowana z etykietą? Jeśli na nieostrych zdjęciach rzadziej
diagnozowano klasę 1 (mikrotętniaki są wtedy niewidoczne), to te zdjęcia mają **zaszumione etykiety** —
DR mógł być, tylko nie dało się go zobaczyć. Model uczony na nich uczy się sprzecznych rzeczy.

In [ ]:
for col in ('sharpness_z', 'mean_brightness', 'contrast'):
    q = pd.qcut(metrics[col], 5, labels=[f'Q{i}' for i in range(1, 6)])
    table = pd.crosstab(q, metrics['level'], normalize='index').round(3)
    table['DR (>0)'] = metrics.groupby(q, observed=True)['binary_label'].mean().round(3)
    print(f'\n{col} (Q1 = najniższe)')
    display(table)

## Szkic reguł odrzucania

Wartości poniżej to **punkt startowy** — ustal je na podstawie siatek z sekcji wyżej
(szczególnie `show_around`) i zapisz tutaj ostateczne.

In [ ]:
THRESHOLDS = {
    'no_fundus':      lambda m: m['mean_brightness'].isna() | (m['mask_frac'] < 0.2),
    'too_dark':       lambda m: m['mean_brightness'] < 25,
    'too_bright':     lambda m: m['mean_brightness'] > 210,
    'low_contrast':   lambda m: m['contrast'] < 8,
    'dark_regions':   lambda m: m['frac_dark'] > 0.3,
    'overexposed':    lambda m: m['frac_bright'] > 0.1,
    'blurry':         lambda m: m['sharpness_z'] < -2.5,
}

flags = pd.DataFrame({name: rule(metrics).fillna(False) for name, rule in THRESHOLDS.items()})
metrics['flags'] = flags.apply(lambda r: ';'.join(r.index[r]), axis=1)
metrics['keep'] = ~flags.any(axis=1)

summary = pd.DataFrame({
    'odrzuconych': flags.sum(),
    'tylko_ta_reguła': (flags & (flags.sum(axis=1) == 1).values[:, None]).sum(),
})
display(summary)

print(f"\nRazem odrzuconych: {(~metrics['keep']).sum()} / {len(metrics)} "
      f"({(~metrics['keep']).mean():.2%})")
display(pd.crosstab(metrics['level'], metrics['keep'], margins=True))

### Losowe próbki odrzuconych — czy reguła łapie to, co powinna?

In [ ]:
rng_seed = 0
for name in THRESHOLDS:
    flagged = metrics[flags[name]]
    show_grid(flagged.sample(min(16, len(flagged)), random_state=rng_seed),
              f'{name}: losowe 16 z {len(flagged)}')

### Losowe próbki zachowanych z dolnych 10% ostrości — czy nie przepuszczamy śmieci?

In [ ]:
kept = metrics[metrics['keep']]
low = kept[kept['sharpness_z'] < kept['sharpness_z'].quantile(0.10)]
show_grid(low.sample(24, random_state=0), 'zachowane, dolne 10% sharpness_z', 'sharpness_z')

## Zapis szkicu flag

Zapisuje do katalogu cache'u (nie do repo). Ostateczny `manifest.csv` z podziałem i flagami
zrobimy osobnym skryptem po ustaleniu progów.

In [ ]:
out_path = CACHE_DIR / 'quality_flags_draft.csv'
metrics[['image', 'level', 'camera_group', 'sharpness_z', 'flags', 'keep']].to_csv(out_path, index=False)
print('zapisano:', out_path)